In [48]:
import sqlite3

In [49]:
import seaborn as sns

In [6]:
conn = sqlite3.connect("truedl.db")

In [8]:
data = conn.execute("""\
select
    tournament_year,
    team_category,
    count(*) as cnt,
    avg(ratio_to_top10) as ratio_avg
from truedl_data
group by tournament_year, team_category
""").fetchall()

In [9]:
data

[('2003', '1001-2000', 1, 0.21161290322580645),
 ('2003', '101-250', 1, 0.5694249649368863),
 ('2003', '2001-3000', 1, 0.23225806451612904),
 ('2003', '251-500', 1, 0.4319775596072932),
 ('2003', '501-1000', 1, 0.3915461624026696),
 ('2003', '51-100', 1, 0.8279569892473119),
 ('2015', '1001-2000', 31, 0.294600701742776),
 ('2015', '101-250', 30, 0.6660233969072186),
 ('2015', '11-25', 16, 0.8813365280197237),
 ('2015', '2001-3000', 15, 0.16913737359913925),
 ('2015', '251-500', 31, 0.5524080164785603),
 ('2015', '26-50', 17, 0.8168961640130528),
 ('2015', '3001-5000', 6, 0.16743572012775365),
 ('2015', '5001+', 24, 0.2881441198857821),
 ('2015', '501-1000', 31, 0.41752683471344443),
 ('2015', '51-100', 25, 0.758304234648222),
 ('2016', '1001-2000', 81, 0.3050830050353893),
 ('2016', '101-250', 83, 0.6666338276078104),
 ('2016', '11-25', 36, 0.864267950666717),
 ('2016', '2001-3000', 46, 0.20645326208181616),
 ('2016', '251-500', 83, 0.5582531055274467),
 ('2016', '26-50', 56, 0.8123913

In [23]:
cat_dict = {
    "11-25": "1. 11-25",
    "26-50": "2. 26-50",
    "51-100": "3. 51-100",
    "101-250": "4. 101-250",
    "251-500": "5. 251-500",
    "501-1000": "6. 501-1000",
    "1001-2000": "7. 1001-2000",
    "2001-3000": "8. 2001-3000",
    "3001-5000": "9. 3001-5000",
    "5001+": "A. 5001+",
}

data_for_plot = []
for x in data:
    data_point = {
        "year": int(x[0]),
        "category": cat_dict[x[1]],
        "ratio": x[3]
    }
    if data_point["year"] > 2003:
        data_for_plot.append(data_point)
data_for_plot = sorted(data_for_plot, key=lambda x: (x["year"], x["category"]))

In [24]:
import pandas as pd
df = pd.DataFrame(data_for_plot)

In [25]:
df

,year,category,ratio
0,2015,1. 11-25,0.881337
1,2015,2. 26-50,0.816896
2,2015,3. 51-100,0.758304
3,2015,4. 101-250,0.666023
4,2015,5. 251-500,0.552408
...,...,...,...
70,2022,4. 101-250,0.737477
71,2022,5. 251-500,0.632544
72,2022,6. 501-1000,0.504212
73,2022,7. 1001-2000,0.384705


In [27]:
%pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 6.0 MB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
import plotly.express as px

In [31]:
fig = px.line(
    df,
    x="year",
    y="ratio",
    color="category"
)
fig.write_html('ratios.html', auto_open=True)

In [65]:
conn = sqlite3.connect("truedl_tournaments.db")

In [66]:
cur = conn.cursor()

In [57]:
cur.execute("drop table truedl")
conn.commit()

In [67]:
cur.execute("""\
create table truedl as
select
    tournament_id,
    min(tournament_name) as tournament_name,
    min(tournament_type) as tournament_type,
    count(*) as cnt,
    round(avg(truedl),1) as truedl
from truedl_tournament_data
group by tournament_id;
""")

In [68]:
conn.commit()

In [72]:
top50 = cur.execute("""\
select * from truedl
order by truedl desc
limit 100;
""").fetchall()

In [73]:
top50

[(4943, 'Первая лига ЧТ', 'Обычный', 4, 10.0),
 (5282, 'Чемпионат Санкт-Петербурга. Высшая лига', '(R) Обычный', 12, 10.0),
 (5283, 'Чемпионат Санкт-Петербурга. Первая лига', '(R) Обычный', 46, 10.0),
 (6149, 'Чемпионат Санкт-Петербурга. Первая лига', '(R) Обычный', 38, 10.0),
 (8688, '30 under 30 (УКР)', 'Синхрон', 1, 10.0),
 (8219, 'Чемпионат Грузии-2022', 'Асинхрон', 10, 9.3),
 (8225, 'Karjalaisia Arvoituksia', 'Асинхрон', 6, 9.3),
 (4664, 'Часы Судного Дня: Полночь', '(R) Синхрон', 73, 9.2),
 (6638, 'Узнать за 60 секунд!', 'Асинхрон', 1, 9.2),
 (4426, 'Посох Аарона', '(R) Синхрон', 94, 9.0),
 (4583, 'Регадо - Б', '(R) Синхрон', 8, 9.0),
 (3915, 'Посох Аарона', '(R) Синхрон', 130, 8.9),
 (4147, 'Гран-при Бауманки. Синхрон', 'Общий зачёт', 71, 8.7),
 (4693, 'Посошок Арончика', 'Синхрон', 68, 8.7),
 (4781, 'Занавес неведения', '(R) Синхрон', 59, 8.7),
 (8209, 'Кубок Гармонии. 1 сезон. Зачёт асинхронов', 'Общий зачёт', 122, 8.7),
 (4029, 'Мстители: Монос Аарона', '(R) Синхрон', 142, 8.

In [71]:
cur.execute("""\
select * from truedl
where tournament_name = "Conversation 16"
order by truedl desc
limit 50;
""").fetchall()

[(6290, 'Conversation 16', 'Асинхрон', 46, 7.7)]

In [41]:
ovsch = cur.execute("""\
select * from truedl
where tournament_name like "%ОВСЧ%"
order by truedl desc
limit 50;
""").fetchall()

In [42]:
ovsch

[(4366, 'ОВСЧ. 3 этап', 799, 6.6),
 (3458, 'ОВСЧ. 4 тур', 789, 5.8),
 (3866, 'ОВСЧ. 1 этап', 834, 5.5),
 (3871, 'ОВСЧ. 6 этап', 760, 5.5),
 (3868, 'ОВСЧ. 3 этап', 807, 5.3),
 (3451, 'ОВСЧ. 2 тур', 857, 5.0),
 (3459, 'ОВСЧ. 5 тур', 781, 5.0),
 (4369, 'ОВСЧ. 6 этап', 810, 5.0),
 (4985, 'ОВСЧ. 5 этап', 841, 5.0),
 (7372, 'ОВСЧ. 3 этап (синхрон)', 419, 4.8),
 (7696, 'ОВСЧ. 1 этап (синхрон)', 494, 4.8),
 (3870, 'ОВСЧ. 5 этап', 838, 4.6),
 (4367, 'ОВСЧ. 4 этап', 891, 4.6),
 (4983, 'ОВСЧ. 3 этап', 843, 4.5),
 (7700, 'ОВСЧ. 3 этап (синхрон)', 549, 4.5),
 (3457, 'ОВСЧ. 3 тур', 813, 4.4),
 (3869, 'ОВСЧ. 4 этап', 688, 4.4),
 (3867, 'ОВСЧ. 2 этап', 851, 3.8),
 (4368, 'ОВСЧ. 5 этап', 777, 3.8),
 (4981, 'ОВСЧ. 1 этап', 1000, 3.8),
 (4986, 'ОВСЧ. 6 этап', 841, 3.7),
 (4364, 'ОВСЧ. 1 этап', 918, 3.6),
 (7698, 'ОВСЧ. 2 этап (синхрон)', 528, 3.6),
 (4982, 'ОВСЧ. 2 этап', 913, 3.5),
 (7376, 'ОВСЧ. 5 этап (синхрон)', 479, 3.5),
 (7368, 'ОВСЧ. 1 этап (синхрон)', 515, 3.4),
 (3460, 'ОВСЧ. 6 тур', 741, 3.3),

In [43]:
len(ovsch)

39

In [46]:
kubbes = cur.execute("""\
select * from truedl
where (tournament_name like "%Кубок Бесконечности%"
or tournament_name like "%Кубок бесконечности%")
order by truedl desc
limit 50;
""").fetchall()

In [47]:
kubbes

[(8446, 'Кубок Бесконечности: четвёртый этап (синхрон)', 104, 6.4),
 (8264, 'Кубок Бесконечности: второй этап (синхрон)', 114, 6.1),
 (8260, 'Кубок Бесконечности: первый этап (синхрон)', 188, 5.6),
 (8442, 'Кубок Бесконечности: третий этап (синхрон)', 131, 5.2),
 (5764, 'Кубок Бесконечности. I этап', 244, 4.9),
 (5766, 'Кубок Бесконечности. III этап', 273, 4.6),
 (6041, 'Кубок Бесконечности. V этап', 357, 4.6),
 (6042, 'Кубок Бесконечности. VI этап', 291, 4.4),
 (5765, 'Кубок Бесконечности. II этап', 193, 4.2),
 (5767, 'Кубок Бесконечности. IV этап', 235, 3.9)]